# Atividade I — Cálculo Numérico
## Erros numéricos em biofluidodinâmica

**Profa. Raquel Jahara Lobosco**

Integrantes e divisão de tarefas:

- _Felipe Marinho de Figueiredo_ — _Parte A B C_
- _Bruno de Carvalho Morais_ — _Parte D E F_

## Preparação

Importamos as bibliotecas e criamos as duas funções de apoio: uma para truncar e outra para calcular os erros.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Circle

In [ ]:
# Trunca um número em "casas" casas decimais (corta sem arredondar).
def truncar(numero, casas):
    fator = 10 ** casas
    return math.trunc(numero * fator) / fator

# Calcula erro absoluto, relativo e percentual.
def calcular_erros(referencia, aproximacao):
    erro_abs = abs(referencia - aproximacao)
    if referencia == 0:
        erro_rel = float("nan")   # não existe erro relativo quando a referência é 0
    else:
        erro_rel = erro_abs / abs(referencia)
    erro_perc = 100 * erro_rel
    return erro_abs, erro_rel, erro_perc

## Parte A — Truncamento e arredondamento

Valores: x1 = 3,14159265, x2 = 98,76543, x3 = 0,00498765.
Para cada um usamos n = 0, 1, 2, 3, 4 casas e calculamos os erros.

In [ ]:
valores = {"x1": 3.14159265, "x2": 98.76543, "x3": 0.00498765}

print("valor | n | truncado | arredondado | Ea(trunc) | Ea(arred)")
for nome, x in valores.items():
    for n in range(5):
        xt = truncar(x, n)
        xr = round(x, n)
        ea_t, er_t, ep_t = calcular_erros(x, xt)
        ea_a, er_a, ep_a = calcular_erros(x, xr)
        print(f"{nome} | {n} | {xt:.5f} | {xr:.5f} | {ea_t:.2e} | {ea_a:.2e}")

### Gráfico do erro absoluto em função de n (escala log)

In [ ]:
for nome, x in valores.items():
    ns = range(5)
    ea_trunc = [calcular_erros(x, truncar(x, n))[0] for n in ns]
    ea_arred = [calcular_erros(x, round(x, n))[0] for n in ns]
    plt.semilogy(ns, ea_trunc, "o-", label=f"{nome} truncamento")
    plt.semilogy(ns, ea_arred, "s--", label=f"{nome} arredondamento")

plt.xlabel("n (casas decimais)")
plt.ylabel("erro absoluto")
plt.title("Erro absoluto x n")
plt.legend(fontsize=8)
plt.show()

**Arredondar sempre gera erro menor ou igual ao truncamento?**

Sim, menor ou igual. Olhando a tabela, o erro do arredondamento nunca é maior que o do truncamento. Às vezes eles são iguais (quando o algarismo cortado é menor que 5) e às vezes o arredondamento é melhor (quando é 5 ou mais). Isso acontece porque o arredondamento escolhe o número mais próximo, enquanto o truncamento sempre corta para baixo.

## Parte B — Medidas de pressão arterial

Dez medidas em mmHg. A referência é a média usando todos os algarismos.
Criamos quatro séries e calculamos as estatísticas de cada uma.

In [ ]:
med = [121.7, 120.4, 122.1, 119.8, 121.2, 120.9, 122.4, 121.0, 120.2, 121.5]

originais   = np.array(med)
arred_int   = np.array([round(v) for v in med])
truncados   = np.array([truncar(v, 0) for v in med])
arred_1casa = np.array([round(v, 1) for v in med])

series = {
    "originais": originais,
    "arred inteiro": arred_int,
    "truncado inteiro": truncados,
    "arred 1 casa": arred_1casa,
}

media_ref = originais.mean()
print(f"Média de referência: {media_ref:.4f} mmHg")
print()
print("série | média(mmHg) | mín(mmHg) | máx(mmHg) | amplitude(mmHg) | desvio(mmHg) | Ea(mmHg) | E%")
for nome, s in series.items():
    media = s.mean()
    minimo = s.min()
    maximo = s.max()
    amplitude = maximo - minimo
    desvio = s.std(ddof=1)   # desvio-padrão amostral
    ea, er, ep = calcular_erros(media_ref, media)
    print(f"{nome} | {media:.4f} | {minimo:.1f} | {maximo:.1f} | {amplitude:.1f} | {desvio:.4f} | {ea:.4f} | {ep:.4f}")

### Gráficos

In [ ]:
# 1) as dez medições nas quatro séries
x = range(1, 11)
for nome, s in series.items():
    plt.plot(x, s, "o-", label=nome)
plt.xlabel("medição")
plt.ylabel("pressão (mmHg)")
plt.title("Dez medições")
plt.legend(fontsize=8)
plt.show()

In [ ]:
# 2) e 3) erros absolutos e percentuais das médias
nomes = list(series.keys())
ea_lista = [calcular_erros(media_ref, series[n].mean())[0] for n in nomes]
ep_lista = [calcular_erros(media_ref, series[n].mean())[2] for n in nomes]

plt.bar(nomes, ea_lista)
plt.ylabel("erro absoluto (mmHg)")
plt.title("Erro absoluto da média")
plt.xticks(rotation=20)
plt.show()

plt.bar(nomes, ep_lista)
plt.ylabel("erro percentual (%)")
plt.title("Erro percentual da média")
plt.xticks(rotation=20)
plt.show()

In [ ]:
# 4) histograma dos originais e dos inteiros arredondados
plt.hist(originais, bins=6, alpha=0.6, label="originais")
plt.hist(arred_int, bins=6, alpha=0.6, label="inteiros arred")
plt.xlabel("pressão (mmHg)")
plt.ylabel("frequência")
plt.title("Histograma")
plt.legend()
plt.show()

**A representação inteira preserva a média e a variabilidade?**

Arredondar para inteiro mantém a média bem próxima da referência, porque os arredondamentos para cima e para baixo quase se cancelam. Truncar para inteiro puxa tudo para baixo e afasta mais a média. Nos dois casos inteiros a variabilidade fica um pouco distorcida, porque a diferença real entre as medidas é de décimos e o inteiro perde essa informação. Ou seja: para a média o inteiro arredondado serve, mas para a variabilidade ele não é tão bom. (Sem interpretação médica.)

## Parte C — Vazão em tubo pequeno (Hagen–Poiseuille)

$$Q = \frac{\pi r^4 \Delta p}{8 \mu L}$$

Com r = 0,80 mm, L = 0,20 m, µ = 3,5e-3 Pa·s, Δp = 1200 Pa.

In [ ]:
# Fórmula de Hagen-Poiseuille (r em metros, Q em m^3/s).
def vazao(r, dp, mu, L):
    return math.pi * r**4 * dp / (8 * mu * L)

r  = 0.80e-3    # m
L  = 0.20       # m
mu = 3.5e-3     # Pa.s
dp = 1200       # Pa

Q_ref = vazao(r, dp, mu, L)
print(f"Q de referência = {Q_ref:.4e} m^3/s")

### Precisão das variáveis

In [ ]:
r_mm = 0.80   # raio em mm, para arredondar/truncar em 1 casa

# aproximações de cada variável
r_ar = round(r_mm, 1) * 1e-3
r_tr = truncar(r_mm, 1) * 1e-3
mu_ar = round(mu, 3)
mu_tr = truncar(mu, 3)
dp_ar = round(dp / 100) * 100
dp_tr = truncar(dp / 100, 0) * 100

cenarios = {
    "1) todos originais": vazao(r, dp, mu, L),
    "2) r arred":         vazao(r_ar, dp, mu, L),
    "2) r trunc":         vazao(r_tr, dp, mu, L),
    "3) mu arred":        vazao(r, dp, mu_ar, L),
    "3) mu trunc":        vazao(r, dp, mu_tr, L),
    "4) dp arred":        vazao(r, dp_ar, mu, L),
    "4) dp trunc":        vazao(r, dp_tr, mu, L),
    "5) todas arred":     vazao(r_ar, dp_ar, mu_ar, L),
    "5) todas trunc":     vazao(r_tr, dp_tr, mu_tr, L),
}

print("cenário | Q(m^3/s) | Ea(m^3/s) | E%")
for nome, q in cenarios.items():
    ea, er, ep = calcular_erros(Q_ref, q)
    print(f"{nome} | {q:.4e} | {ea:.2e} | {ep:.4f}")

> Alguns cenários dão erro zero porque a aproximação não muda o valor: 0,80 arredondado ou truncado em 1 casa continua 0,80, e 1200 em centenas continua 1200. Quem muda de verdade é a viscosidade (0,0035 vira 0,004 arredondando e 0,003 truncando), por isso ela gera os maiores erros.

### Sensibilidade ao raio (0,70 a 0,90 mm)

In [ ]:
raios = np.linspace(0.70, 0.90, 21)   # em mm

Q_r  = np.array([vazao(rr*1e-3, dp, mu, L) for rr in raios])
Q_ar = np.array([vazao(round(rr,1)*1e-3, dp, mu, L) for rr in raios])
Q_tr = np.array([vazao(truncar(rr,1)*1e-3, dp, mu, L) for rr in raios])

ea_ar = np.abs(Q_r - Q_ar)
er_ar = ea_ar / Q_r

# gráfico 1: Q em função de r
plt.plot(raios, Q_r, label="Q referência")
plt.plot(raios, Q_ar, "--", label="r arredondado")
plt.xlabel("r (mm)"); plt.ylabel("Q (m^3/s)")
plt.title("Q em função de r"); plt.legend(); plt.show()

# gráfico 2: erro absoluto
plt.plot(raios, ea_ar)
plt.xlabel("r (mm)"); plt.ylabel("erro absoluto (m^3/s)")
plt.title("Erro absoluto de Q"); plt.show()

# gráfico 3: erro relativo
plt.plot(raios, er_ar)
plt.xlabel("r (mm)"); plt.ylabel("erro relativo")
plt.title("Erro relativo de Q"); plt.show()

# gráfico 4: erro percentual
plt.plot(raios, 100*er_ar)
plt.xlabel("r (mm)"); plt.ylabel("erro percentual (%)")
plt.title("Erro percentual de Q"); plt.show()

**Por que o erro no raio é amplificado na vazão?**

Porque Q depende de r elevado à quarta potência. Quando o raio tem um erro relativo pequeno, esse erro aparece multiplicado por 4 na vazão (dQ/Q ≈ 4·dr/r). Então um erro de 1% no raio vira cerca de 4% na vazão. O expoente 4 é o motivo da amplificação.

## Parte D — Nanocateter

r = 50 nm, L = 10 µm, µ = 3,5e-3 Pa·s, Δp = 5000 Pa.

In [ ]:
r_n  = 50e-9     # m
L_n  = 10e-6     # m
mu_n = 3.5e-3    # Pa.s
dp_n = 5000      # Pa

Q_n = vazao(r_n, dp_n, mu_n, L_n)
print(f"Q do nanocateter = {Q_n:.4e} m^3/s")

### Erro geométrico (Δr de 0,1 a 5,0 nm)

In [ ]:
drs = np.arange(0.1, 5.01, 0.1) * 1e-9   # em metros

ea_max = []
er_max = []
Q_menos = []
Q_mais = []
for dr in drs:
    qm = vazao(r_n - dr, dp_n, mu_n, L_n)
    qp = vazao(r_n + dr, dp_n, mu_n, L_n)
    Q_menos.append(qm)
    Q_mais.append(qp)
    ea = max(abs(qm - Q_n), abs(qp - Q_n))
    ea_max.append(ea)
    er_max.append(ea / Q_n)

drs_nm = drs * 1e9

# Q(r-), Q(r), Q(r+)
plt.plot(drs_nm, Q_menos, label="Q(r-Δr)")
plt.axhline(Q_n, color="k", label="Q(r)")
plt.plot(drs_nm, Q_mais, label="Q(r+Δr)")
plt.xlabel("Δr (nm)"); plt.ylabel("Q (m^3/s)")
plt.title("Q para r-, r e r+"); plt.legend(); plt.show()

# erro absoluto máximo
plt.plot(drs_nm, ea_max)
plt.xlabel("Δr (nm)"); plt.ylabel("erro absoluto (m^3/s)")
plt.title("Erro absoluto máximo de Q"); plt.show()

# erro relativo máximo
plt.plot(drs_nm, er_max)
plt.xlabel("Δr (nm)"); plt.ylabel("erro relativo")
plt.title("Erro relativo máximo de Q"); plt.show()

# erro percentual máximo
plt.plot(drs_nm, [100*e for e in er_max])
plt.xlabel("Δr (nm)"); plt.ylabel("erro percentual (%)")
plt.title("Erro percentual máximo de Q"); plt.show()

In [ ]:
# Comparação com a aproximação linear 4*Δr/r
er_linear = 4 * drs / r_n
plt.plot(drs_nm, [100*e for e in er_max], "o-", ms=3, label="erro direto")
plt.plot(drs_nm, 100*er_linear, "--", label="aprox linear 4·Δr/r")
plt.xlabel("Δr (nm)"); plt.ylabel("erro percentual (%)")
plt.title("Erro direto x aproximação linear"); plt.legend(); plt.show()

### Precisão computacional (float32 x float64)

In [ ]:
r32 = np.float32(r_n)
r64 = np.float64(r_n)

def vazao_np(r, dp, mu, L):
    return np.pi * r**4 * dp / (8 * mu * L)

Q32 = vazao_np(r32, np.float32(dp_n), np.float32(mu_n), np.float32(L_n))
Q64 = vazao_np(r64, np.float64(dp_n), np.float64(mu_n), np.float64(L_n))

print(f"r^4 float32 = {np.float32(r32**4):.4e}")
print(f"r^4 float64 = {r64**4:.4e}")
print(f"Q float32 = {float(Q32):.4e}")
print(f"Q float64 = {float(Q64):.4e}")
print(f"diferença absoluta = {abs(float(Q32)-float(Q64)):.2e}")
print(f"diferença relativa = {abs(float(Q32)-float(Q64))/abs(float(Q64)):.2e}")
print(f"eps float32 = {np.finfo(np.float32).eps:.2e}")
print(f"eps float64 = {np.finfo(np.float64).eps:.2e}")

In [ ]:
# Variação de vazão de duas formas, para Δr/r cada vez menor
print("Δr/r | delta1 (subtração) | delta2 (fatorada)")
for razao in [1e-1, 1e-3, 1e-5, 1e-7]:
    dr = razao * r_n
    delta1 = (vazao(r_n + dr, dp_n, mu_n, L_n) - Q_n) / Q_n
    delta2 = (1 + dr/r_n)**4 - 1
    print(f"{razao:.0e} | {delta1:.6e} | {delta2:.6e}")

As duas fórmulas deveriam dar o mesmo resultado. Para Δr/r muito pequeno, delta1 subtrai dois números quase iguais e os algarismos significativos se perdem (cancelamento). Em float64 isso só aparece para valores bem pequenos; em float32 apareceria bem antes, porque o eps é maior.

### Fenômeno físico

Quando a escala diminui para nanômetros, a razão entre a superfície e o volume aumenta (ela cresce como 1/r). Isso faz os efeitos de viscosidade e de parede ficarem muito mais importantes do que em um tubo grande. O modelo de Hagen–Poiseuille supõe escoamento laminar ideal e sem deslizamento na parede, o que fica pouco realista nessa escala. Por isso, um resultado pode ter muitas casas decimais e ser preciso na conta, mas ainda assim representar mal a física real: precisão aritmética e validade do modelo são coisas diferentes.

## Parte E — Quadro único de resultados (PNG)

Um único painel reunindo os principais valores, a comparação truncamento/arredondamento, os erros, a sensibilidade ao raio, a comparação float32/float64 e uma conclusão curta.

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Erros numéricos em biofluidodinâmica", fontsize=16, fontweight="bold")

# cartões com os principais valores
axs[0,0].axis("off")
texto = (f"Q tubo = {Q_ref:.2e} m3/s\n\n"
         f"Q nano = {Q_n:.2e} m3/s\n\n"
         f"dp tubo = {dp} Pa\n"
         f"dp nano = {dp_n} Pa")
axs[0,0].text(0.05, 0.9, texto, va="top", fontsize=13,
              bbox=dict(boxstyle="round", fc="#EAF2FB"))
axs[0,0].set_title("Principais valores")

# truncamento x arredondamento (x1)
ns = range(5)
ea_t = [calcular_erros(3.14159265, truncar(3.14159265, n))[0] for n in ns]
ea_a = [calcular_erros(3.14159265, round(3.14159265, n))[0] for n in ns]
axs[0,1].semilogy(ns, ea_t, "o-", label="trunc")
axs[0,1].semilogy(ns, ea_a, "s--", label="arred")
axs[0,1].set_title("Trunc x Arred (x1)")
axs[0,1].set_xlabel("n"); axs[0,1].legend()

# os três erros (usando as médias da Parte B)
axs[0,2].bar(nomes, ep_lista)
axs[0,2].set_title("Erro % das médias (Parte B)")
axs[0,2].tick_params(axis="x", rotation=25, labelsize=8)

# sensibilidade de Q ao raio
axs[1,0].plot(raios, Q_r)
axs[1,0].set_title("Q em função de r (tubo)")
axs[1,0].set_xlabel("r (mm)"); axs[1,0].set_ylabel("Q (m3/s)")

# float32 x float64
axs[1,1].axis("off")
texto2 = (f"Q float32 = {float(Q32):.4e}\n\n"
          f"Q float64 = {float(Q64):.4e}\n\n"
          f"eps 32 = {np.finfo(np.float32).eps:.1e}\n"
          f"eps 64 = {np.finfo(np.float64).eps:.1e}")
axs[1,1].text(0.05, 0.9, texto2, va="top", fontsize=12, family="monospace",
              bbox=dict(boxstyle="round", fc="#EAFBEA"))
axs[1,1].set_title("float32 x float64")

# conclusão
axs[1,2].axis("off")
conclusao = ("Conclusão: como Q depende de r elevado à quarta,\n"
             "um erro pequeno no raio vira um erro 4x maior\n"
             "na vazão. No nanocateter isso é crítico.\n"
             "Precisão na conta não garante que o modelo\n"
             "represente bem a física real.")
axs[1,2].text(0.05, 0.9, conclusao, va="top", fontsize=11,
              bbox=dict(boxstyle="round", fc="#FFF6E5"))
axs[1,2].set_title("Nanocateter")

plt.tight_layout()
plt.savefig("quadro_sintese.png")
plt.show()
print("Quadro salvo em quadro_sintese.png")

## GIF demonstrativo

O raio do nanocateter varia de 45 a 55 nm. Cada quadro mostra a seção do canal, o raio e a vazão atuais, o erro percentual em relação a r = 50 nm e a curva Q x r sendo construída.

In [ ]:
raios_gif = np.linspace(45e-9, 55e-9, 40)
Q_gif = np.array([vazao(rr, dp_n, mu_n, L_n) for rr in raios_gif])
Q50 = vazao(50e-9, dp_n, mu_n, L_n)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5))

def atualizar(i):
    ax1.clear(); ax2.clear()
    rr = raios_gif[i]
    q = Q_gif[i]
    ep = 100 * abs(q - Q50) / Q50

    # seção transversal do canal
    ax1.add_patch(Circle((0, 0), rr*1e9, color="#4C72B0", alpha=0.5))
    ax1.add_patch(Circle((0, 0), 50, fill=False, ls="--"))
    ax1.set_xlim(-60, 60); ax1.set_ylim(-60, 60)
    ax1.set_aspect("equal")
    ax1.set_xlabel("nm")
    ax1.set_title("Seção do canal")
    ax1.text(-58, 52, f"r = {rr*1e9:.1f} nm\nQ = {q:.2e}\nE% = {ep:.1f}%",
             va="top", fontsize=9,
             bbox=dict(boxstyle="round", fc="white"))

    # curva Q x r construída aos poucos
    ax2.plot(raios_gif[:i+1]*1e9, Q_gif[:i+1], "o-", ms=3)
    ax2.set_xlim(44, 56)
    ax2.set_ylim(Q_gif.min()*0.95, Q_gif.max()*1.05)
    ax2.set_xlabel("r (nm)"); ax2.set_ylabel("Q (m^3/s)")
    ax2.set_title("Curva Q x r")

anim = animation.FuncAnimation(fig, atualizar, frames=len(raios_gif), interval=120)
anim.save("nanocateter.gif", writer=animation.PillowWriter(fps=8))
plt.close(fig)
print("GIF salvo em nanocateter.gif")

# mostra o GIF no Colab
from IPython.display import Image
Image(filename="nanocateter.gif")

## Conclusão

Nesta atividade vimos que todo resultado numérico tem um erro que precisa ser medido. O arredondamento dá erro sempre menor ou igual ao do truncamento. Na fórmula de Hagen–Poiseuille, como Q depende de r elevado à quarta potência, um erro no raio é multiplicado por 4 na vazão, e isso fica muito importante na escala do nanocateter. Também vimos que usar float32 ou float64 muda quando aparece a perda de precisão por cancelamento. No fim, a lição principal é que ter muitas casas decimais não garante que o modelo represente bem a física: precisão da conta e validade do modelo são coisas diferentes.